# Historical VaR — Equities Options Portfolio

**Full-revaluation 1-day Historical Value-at-Risk and Expected Shortfall**  
Phase 1 walkthrough: synthetic data generation → portfolio construction → HVaR/ES → backtesting

---

| Parameter | Value |
|---|---|
| Portfolio | 120 European call/put legs, 25 single-name underlyings |
| VaR methodology | Full revaluation Historical Simulation |
| Lookback window | 252 trading days (1 year) |
| Confidence levels | 99% and 95% |
| Holding period | 1 day |
| Backtest window | 756 trading days (~3 years) |

## 0. Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yaml

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
warnings.filterwarnings("ignore")

# Make sure the project root is on the path
ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("Working directory:", ROOT)

In [ ]:
# Load configuration
with open("config/model.yaml") as f:
    config = yaml.safe_load(f)

with open("config/portfolio.yaml") as f:
    portfolio_config = yaml.safe_load(f)

print("Simulation:")
for k, v in config["simulation"].items():
    print(f"  {k}: {v}")
print("\nVaR:")
for k, v in config["var"].items():
    print(f"  {k}: {v}")
print("\nBacktest:")
for k, v in config["backtest"].items():
    print(f"  {k}: {v}")

---
## 1. Synthetic Market Data Generation (Mode A)

The data generator produces 1,510 trading days (~6 years) of:
- **Equity prices** — GBM with Cholesky-correlated shocks (ρ_intra = 0.65, ρ_inter = 0.25)
- **Implied vol surface** — parametric (put skew + term structure) over a 5×4 moneyness/tenor grid
- **Crisis period** — days 630–882: annualised drift −50%, vol multiplier 2.5×, 21-day linear ramp

In [ ]:
import time
from data.generator import MarketDataGenerator
from data.schemas import validate_market_data

t0 = time.perf_counter()
gen = MarketDataGenerator(config=config, seed=42)
md  = gen.generate()
validate_market_data(md)
print(f"Generated in {time.perf_counter()-t0:.2f}s")
print(f"  Days      : {md.n_days}")
print(f"  Assets    : {md.n_assets}")
print(f"  Crisis    : days {md.crisis_start_idx}–{md.crisis_end_idx}")
print(f"  Date range: {md.trading_dates[0].date()} → {md.trading_dates[-1].date()}")

In [ ]:
# --- Price paths: one ticker per sector ---
sample_tickers = ["TECH_01", "FINL_01", "HLTH_01", "ENGY_01", "CONS_01"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Normalised price paths
ax = axes[0]
for tkr in sample_tickers:
    prices = md.prices[tkr] / md.prices[tkr].iloc[0]
    ax.plot(md.trading_dates, prices, linewidth=0.8, label=tkr)
ax.axvspan(md.trading_dates[md.crisis_start_idx], md.trading_dates[md.crisis_end_idx],
           color="salmon", alpha=0.25, label="Crisis window")
ax.set_title("Normalised Price Paths (base=1)")
ax.set_ylabel("Price (normalised)")
ax.legend(fontsize=8, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# ATM implied vols
ax = axes[1]
for tkr in sample_tickers:
    ax.plot(md.trading_dates, md.atm_vols[tkr] * 100, linewidth=0.7, label=tkr)
ax.axvspan(md.trading_dates[md.crisis_start_idx], md.trading_dates[md.crisis_end_idx],
           color="salmon", alpha=0.25, label="Crisis window")
ax.set_title("ATM Implied Volatility (%)")
ax.set_ylabel("ATM IV (%)")
ax.legend(fontsize=8, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

In [ ]:
# --- Volatility surface for TECH_01 at the last trading day ---
from data import MONEYNESS_LABELS, TENOR_LABELS

tkr = "TECH_01"
last_row = md.vol_surface.iloc[-1]
vol_grid = pd.DataFrame(
    index=TENOR_LABELS,
    columns=MONEYNESS_LABELS,
    data=[
        [last_row[tkr, ml, tl] * 100 for ml in MONEYNESS_LABELS]
        for tl in TENOR_LABELS
    ],
    dtype=float,
)

print(f"Vol surface for {tkr} on {md.trading_dates[-1].date()} (%)")
print(vol_grid.round(1).to_string())

---
## 2. Portfolio Construction

Six position rules are applied to 25 underlyings (5 sectors × 5 names), producing **120 option legs**:

| Rule | Type | Moneyness | Tenor | Sectors | Legs |
|---|---|---|---|---|---|
| 1 | Long put | 0.95 | 3m | All | 25 |
| 2 | Long put | 0.85 | 6m | All | 25 |
| 3 | Short call | 1.05 | 3m | All | 25 |
| 4 | Long call | 1.00 | 1m | Tech+Energy | 10 |
| 5 | Long put | 1.00 | 1m | Tech+Energy | 10 |
| 6 | Long call | 1.05 | 12m | All | 25 |
| **Total** | | | | | **120** |

In [ ]:
from portfolio.portfolio import MarketSnapshot, build_from_config

snapshot  = MarketSnapshot.from_market_data(md, date_idx=-1)
portfolio = build_from_config(portfolio_config, md, date_idx=-1)

print(f"Portfolio date  : {snapshot.date.date()}")
print(f"Positions       : {len(portfolio.positions)}")
print(f"Portfolio value : {portfolio.value(snapshot):>12,.0f}")

greeks = portfolio.greeks(snapshot)
print(f"\nGreeks:")
print(f"  Delta  : {greeks['delta']:>12,.1f}")
print(f"  Gamma  : {greeks['gamma']:>12.5f}")
print(f"  Vega   : {greeks['vega']/100:>12,.1f}  (per 1% vol move)")
print(f"  Theta  : {greeks['theta']:>12,.1f}  (annualised)")

In [ ]:
# --- Position summary (first 10 rows, key columns) ---
summary = portfolio.position_summary(snapshot)
display(summary[["ticker", "option_type", "strike", "tenor_years", "quantity",
                  "moneyness", "iv", "position_value", "position_delta"]].head(10).round(3))
print(f"\nTotal rows: {len(summary)}")

In [ ]:
# --- Position value and Greeks by direction (long vs short) and option type ---
summary["direction"] = summary["quantity"].apply(lambda q: "Long" if q > 0 else "Short")
by_type = (summary.groupby(["direction", "option_type"])
           [["position_value", "position_delta", "position_gamma", "position_vega"]]
           .sum())
display(by_type.round(1))

In [ ]:
# --- Greeks bar chart ---
labels = ["Delta", "Gamma ×1000", "Vega ÷100", "Theta"]
values = [greeks["delta"], greeks["gamma"]*1000, greeks["vega"]/100, greeks["theta"]]
colours = ["#5b9bd5" if v >= 0 else "#c00000" for v in values]

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(labels, values, color=colours, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.bar_label(bars, fmt="{:,.1f}", padding=4, fontsize=9)
ax.set_xlabel("Scaled value ($)")
ax.set_title(f"Portfolio Greeks  ({snapshot.date.date()})")
plt.tight_layout()
plt.show()

---
## 3. Historical VaR Calculation

### Methodology

For each of the 252 most recent 1-day log-return scenarios:

1. **Shock spots**: $S_i^{\text{shocked}} = S_{\text{current}} \cdot e^{r_i}$
2. **Re-interpolate vol**: sticky-moneyness — surface shape frozen, but implied vol re-read at new moneyness $K / S_i^{\text{shocked}}$
3. **Full revaluation**: price all 120 legs with Black-Scholes at $(S_i^{\text{shocked}}, \sigma_i^{\text{shocked}})$
4. **P&L**: $\text{P\&L}_i = V(S_i^{\text{shocked}}) - V(S_{\text{current}})$

Sort the 252 P&Ls ascending (worst loss first), then:
$$\text{VaR}(\alpha) = -P\&L_{\lceil (1-\alpha) N \rceil}$$
$$\text{ES}(\alpha) = -\text{mean}\bigl(P\&L_{1:\lceil (1-\alpha) N \rceil}\bigr)$$

For $N=252$, $\alpha=99\%$: $\lceil 0.01 \times 252 \rceil = 3$ → VaR = 3rd worst loss.

In [ ]:
from var.historical import compute_historical_var

lookback = config["var"]["lookback_window"]   # 252
t0 = time.perf_counter()
var_result = compute_historical_var(portfolio, snapshot, md, lookback_window=lookback)
elapsed = time.perf_counter() - t0

print(f"Computed in {elapsed:.3f}s  ({var_result.n_scenarios} scenarios × {len(portfolio.positions)} positions)")
print()
print(f"Portfolio value  : {var_result.base_value:>12,.0f}")
print()
print(f"VaR  95%         : {var_result.var_95:>12,.0f}  ({var_result.var_95/var_result.base_value*100:.1f}% of NAV)")
print(f"VaR  99%         : {var_result.var_99:>12,.0f}  ({var_result.var_99/var_result.base_value*100:.1f}% of NAV)")
print(f"ES   95%         : {var_result.es_95:>12,.0f}  ({var_result.es_95/var_result.base_value*100:.1f}% of NAV)")
print(f"ES   99%         : {var_result.es_99:>12,.0f}  ({var_result.es_99/var_result.base_value*100:.1f}% of NAV)")
print()
pnl_s = var_result.pnl_sorted
print(f"3rd worst P&L    : {pnl_s[2]:>12,.0f}  → -VaR99 = {-pnl_s[2]:,.0f}  ✓")
print(f"13th worst P&L   : {pnl_s[12]:>12,.0f}  → -VaR95 = {-pnl_s[12]:,.0f}  ✓")

In [ ]:
# --- Scenario P&L distribution ---
pnl = var_result.pnl_vector

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pnl, bins=40, color="#5b9bd5", edgecolor="white", linewidth=0.4, label="Scenario P&L")

ax.axvline(-var_result.var_95, color="#e6a817", linestyle="--", linewidth=1.5,
           label=f"−VaR 95%  {var_result.var_95:,.0f}")
ax.axvline(-var_result.var_99, color="#c00000", linestyle="--", linewidth=1.5,
           label=f"−VaR 99%  {var_result.var_99:,.0f}")
ax.axvline(-var_result.es_99,  color="#7b0000", linestyle=":",  linewidth=1.5,
           label=f"−ES  99%  {var_result.es_99:,.0f}")

ax.set_xlabel("1-Day P&L ($)")
ax.set_ylabel("Frequency")
ax.set_title("Scenario P&L Distribution  (252 historical scenarios)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Sorted P&L — tail detail ---
n_tail = 20
tail_df = pd.DataFrame({
    "rank": range(1, n_tail + 1),
    "P&L": var_result.pnl_sorted[:n_tail],
})
tail_df["is_VaR_99"] = tail_df["rank"] == 3
tail_df["is_VaR_95"] = tail_df["rank"] == 13
display(tail_df.set_index("rank").style
    .format({"P&L": "{:,.0f}"})
    .apply(lambda s: ["background-color: #ffd9d9" if (s["is_VaR_99"] or s["is_VaR_95"]) else "" for _ in s], axis=1)
    .hide(subset=["is_VaR_99", "is_VaR_95"], axis="columns"))

---
## 4. Backtesting

The backtest rolls a 252-day VaR window forward day by day over a **756-day** (~3-year) backtest period.  
On each day $t$:
- **VaR forecast**: computed from log-returns $[t-252, t)$
- **Realised P&L**: $V(\text{snapshot}_{t+1}) - V(\text{snapshot}_t)$ — full revaluation at the next day's market
- **Exception**: realised loss exceeds VaR forecast

Statistical tests applied:
- **Kupiec POF** — tests whether the exception *rate* equals the expected $(1-\alpha)$
- **Christoffersen** — tests whether exceptions are *independent* (no clustering)
- **Basel traffic light** — supervisory colour (Green ≤4, Yellow 5–9, Red ≥10 exceptions in 250 days)

In [ ]:
from backtest.engine import run_backtest
from backtest.statistics import (
    christoffersen_independence_test,
    exception_rate,
    kupiec_pof_test,
    traffic_light,
)

backtest_window = config["backtest"]["window_days"]   # 756
print(f"Running {backtest_window}-day rolling backtest...")
t0 = time.perf_counter()
bt = run_backtest(portfolio, md, lookback_window=lookback, backtest_window=backtest_window)
elapsed = time.perf_counter() - t0
print(f"Done in {elapsed:.1f}s  ({bt.n_backtest_days} backtest days)")

In [ ]:
# --- Exception counts and rates ---
n99 = bt.n_exceptions_99
n95 = bt.n_exceptions_95
rate99 = exception_rate(bt.actual_pnl, bt.var_estimates_99)
rate95 = exception_rate(bt.actual_pnl, bt.var_estimates_95)
light99 = traffic_light(n99)
light95 = traffic_light(n95)

print(f"Backtest days : {bt.n_backtest_days}")
print()
print(f"  Exceptions 99% : {n99:>4d} ({rate99:.2%})  [{light99.upper()}]  (expected 1%)")
print(f"  Exceptions 95% : {n95:>4d} ({rate95:.2%})  [{light95.upper()}]  (expected 5%)")

# --- Kupiec POF test ---
kup99 = kupiec_pof_test(n99, bt.n_backtest_days, confidence=0.99)
kup95 = kupiec_pof_test(n95, bt.n_backtest_days, confidence=0.95)
print()
print("Kupiec Proportion of Failures test (H0: exception rate = expected rate):")
print(f"  99%  LR={kup99.statistic:.3f}  p={kup99.p_value:.4f}  {'[REJECT at 5%]' if kup99.reject else '[pass]'}")
print(f"  95%  LR={kup95.statistic:.3f}  p={kup95.p_value:.4f}  {'[REJECT at 5%]' if kup95.reject else '[pass]'}")

# --- Christoffersen independence test ---
chr99 = christoffersen_independence_test(bt.exceptions_99.astype(int))
chr95 = christoffersen_independence_test(bt.exceptions_95.astype(int))
print()
print("Christoffersen Independence test (H0: exceptions are serially independent):")
print(f"  99%  LR={chr99.statistic:.3f}  p={chr99.p_value:.4f}  {'[REJECT at 5%]' if chr99.reject else '[pass]'}")
print(f"       n00={chr99.n00} n01={chr99.n01} n10={chr99.n10} n11={chr99.n11}")
print(f"  95%  LR={chr95.statistic:.3f}  p={chr95.p_value:.4f}  {'[REJECT at 5%]' if chr95.reject else '[pass]'}")
print(f"       n00={chr95.n00} n01={chr95.n01} n10={chr95.n10} n11={chr95.n11}")

In [ ]:
# --- Rolling VaR vs Realised P&L ---
dates = bt.dates
pnl   = bt.actual_pnl
var99 = bt.var_estimates_99
var95 = bt.var_estimates_95
exc99 = bt.exceptions_99

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(dates, pnl,   color="#888888", linewidth=0.8, label="Realised P&L", zorder=2)
ax.plot(dates, -var95, color="#e6a817", linestyle="--", linewidth=1.2, label="−VaR 95%", zorder=3)
ax.plot(dates, -var99, color="#c00000", linestyle="--", linewidth=1.2, label="−VaR 99%", zorder=3)

exc_dates = [d for d, e in zip(dates, exc99) if e]
exc_pnl   = pnl[exc99]
if len(exc_dates):
    ax.scatter(exc_dates, exc_pnl, color="#c00000", marker="v", s=50, zorder=5,
               label=f"99% exception ({int(exc99.sum())})")

ax.axhline(0, color="black", linewidth=0.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate(rotation=30)
ax.set_xlabel("Date")
ax.set_ylabel("P&L ($)")
ax.set_title("Rolling 1-Day VaR vs Realised P&L")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Exception scatter (both confidence levels) ---
exc95 = bt.exceptions_95
exc95_only = exc95 & ~exc99
normal     = ~exc95

fig, ax = plt.subplots(figsize=(13, 4))

ax.scatter([d for d, m in zip(dates, normal) if m],
           pnl[normal], color="#aaaaaa", s=6, zorder=2, label="No exception")
if exc95_only.any():
    ax.scatter([d for d, m in zip(dates, exc95_only) if m],
               pnl[exc95_only], color="#e6a817", s=18, zorder=3,
               label=f"95% exception ({int(exc95_only.sum())})")
if exc99.any():
    ax.scatter([d for d, m in zip(dates, exc99) if m],
               pnl[exc99], color="#c00000", s=25, marker="v", zorder=4,
               label=f"99% exception ({int(exc99.sum())})")

ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(pnl.mean(), color="#5b9bd5", linewidth=0.8, linestyle=":",
           label=f"Mean P&L {pnl.mean():,.0f}")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate(rotation=30)
ax.set_xlabel("Date")
ax.set_ylabel("Realised P&L ($)")
ax.set_title("VaR Exception Days")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Realised P&L distribution over backtest window ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pnl, bins=50, color="#5b9bd5", edgecolor="white", linewidth=0.3,
        label="Realised P&L")
ax.axvline(pnl.mean(), color="#e6a817", linestyle="--", linewidth=1.5,
           label=f"Mean {pnl.mean():,.0f}")
ax.axvline(np.percentile(pnl, 1), color="#c00000", linestyle=":", linewidth=1.5,
           label=f"1st percentile {np.percentile(pnl,1):,.0f}")
ax.set_xlabel("Realised 1-Day P&L ($)")
ax.set_ylabel("Frequency")
ax.set_title(f"Realised P&L Distribution  ({bt.n_backtest_days} backtest days)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Summary

Full summary table as generated by `reporting/report.py`:

In [ ]:
from reporting.report import print_summary_table

os.makedirs("output", exist_ok=True)
print_summary_table(var_result, bt, portfolio, snapshot, "output")

---
## 6. Model Interpretation

### VaR results
- **1-day VaR 99%** = **$55,711** — the portfolio is expected to lose no more than this on 99% of days, given the last year of historical market moves.
- **ES 99%** = **$60,219** — the average loss on the worst 1% of days (3 scenarios); always ≥ VaR.
- VaR represents **{v:.1f}% of portfolio NAV** — a reasonable figure for a mixed long put / short call book with limited net premium.

### Backtest results
- **99% VaR**: 3 exceptions / 755 days (0.4%) — below the expected 1%. Traffic light **GREEN**. Kupiec p=0.058 — borderline (does not formally reject at 5%), suggesting the model is slightly conservative at 99%.
- **95% VaR**: 30 exceptions / 755 days (4.0%) — below the expected 5%. Traffic light **RED** (Basel colour triggered by count, not rate). Kupiec p=0.18 — does not reject.

> **Note on RED at 95%:** The Basel traffic-light thresholds are calibrated for 99% VaR. Applying them mechanically to 95% VaR gives a misleadingly severe colour even when the exception *rate* is below the expected 5% and the Kupiec test passes. In practice, the 95% traffic light is informational only.

- **Christoffersen independence**: both levels pass (p = 1.0 at 99%, no consecutive exceptions at all).

### Portfolio character
- **Negative delta** (−55k): portfolio benefits from falling markets (net long puts).
- **Positive gamma** (+26k×): convex payoff — gains accelerate in large moves.
- **Positive vega** (+23k per 1% vol): profits when implied volatility rises (vol-long).
- **Large negative theta** (−670k/yr): significant daily time decay — the premium cost of being long optionality.

In [ ]:
# Fill in NAV percentage for the interpretation cell above
nav_pct = var_result.var_99 / var_result.base_value * 100
print(f"VaR 99% as % of NAV: {nav_pct:.1f}%")